**LangChain: Beginner to advence**

**LangChain Model**

In [ ]:
!pip install -q langchain langchain-core langchain-openai openai langchain-anthropic langchain-google-genai google-generativeai langchain-huggingface transformers huggingface-hub python-dotenv

In [ ]:
import langchain
import langchain_core
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint,HuggingFacePipeline
from sklearn.metrics.pairwise import cosine_similarity
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
import openai
import langchain_anthropic
import langchain_google_genai
import google.generativeai as genai
import langchain_huggingface
import transformers
import huggingface_hub
from dotenv import load_dotenv
import numpy as np
import sklearn

load_dotenv()
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")

llm = OpenAI(model="gpt-3.5-turbo-instruct")
result = llm.invoke("What is the capital of Bangladesh")
print(result)

In [ ]:
model = ChatOpenAI(model="gpt-4", temperature=1.5, max_completion_tokens=10)
result = model.invoke("Write a 5 line poem on cricket")
print(result.content)

In [ ]:
from google.colab import userdata
userdata.get('ANTROPIC_API_KEY')

model = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=1.5, max_tokens=10)
result = model.invoke("Write a 5 star performer of Bangldesh cricket")
print(result.content)

In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=1.5, max_completion_tokens=10)
result = model.invoke("Write a 5 star performer of Bangldesh cricket")
print(result.content)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)
result = model.invoke("Write a 5 star performer of Bangldesh cricket")
print(result.content)

In [ ]:
import os
os.environ['HF_HOME'] = 'D:/huggingface_cache'

llm = HuggingFacePipeline(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs=dict(
        temperature=1.5,
        max_length=100
    )
)

model = ChatHuggingFace(llm=llm)
result = model.invoke("Write a 5 star performer of Bangldesh cricket")
print(result.content)

In [ ]:
embedding = OpenAIEmbeddings(model='text-embedding-3-large', dimensions=32)
result = embedding.embed_query("Hello world")
print(str(result))

In [ ]:
embedding = OpenAIEmbeddings(model='text-embedding-3-large', dimensions=32)
documents = [
    "Dhaka is the capital of Bangladesh.",
    "Cricket is a highly popular sport in South Asia.",
    "LangChain makes developing LLM applications easy."
]
result = embedding.embed_documents(documents)
print(str(result))

In [ ]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
documents = [
    "Dhaka is the capital of Bangladesh.",
    "Cricket is a highly popular sport in South Asia.",
    "LangChain makes developing LLM applications easy."
]
result = embedding.embed_documents(documents)
print(str(result))

In [ ]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
documents = [
    "Dhaka is the capital city of Bangladesh.",
    "Shakib Al Hasan is one of the best all-rounders in cricket.",
    "Python is a versatile programming language."
]
query = "Tell me about Bangladesh's capital"

doc_embeddings = embedding.embed_documents(documents)
query_embeddings = embedding.embed_query(query)

scores = cosine_similarity([query_embeddings], doc_embeddings)[0]
index, score = sorted(list(enumerate(scores)), key=lambda x: x[1])[-1]

print("Query:", query)
print("Most similar document:", documents[index])
print("Similarity score is:", score)

**Prompt in Langchain**

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import streamlit as st
from langchain_core.prompts import PromptTemplate,load_prompt, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()

model = ChatOpenAI(model="gpt-4", temperature=1.5)
result = model.invoke("Write a 5 line poem on cricket")
print(result.content)

In [ ]:
template = PromptTemplate(
    template="""
  Please summarize the research paper titled "{paper_input}" with the following specifications:
  Explanation Style: {style_input}
  Explanation Length: {length_input}
  1. Mathematical Details:
     - Include relevant mathematical equations if present in the paper.
     - Explain the mathematical concepts using simple, intuitive code snippets where applicable.
  2. Analogies:
    - Use relatable analogies to simplify complex ideas.
  If certain information is not available in the paper, respond with: "Insufficient information available" instead of guessing.
  Ensure the summary is clear, accurate, and aligned with the provided style and length.
  """,
  input_variables=['paper_input', 'style_input','length_input'],
  validate_template=True
)

template.save('template.json')

In [ ]:
model = ChatOpenAI()
st.header("Research Tool")
paper_input = st.selectbox( "Select Research Paper Name", ["Attention Is All You Need", "BERT: Pre-training of Deep Bidirectional Transformers", "GPT-3: Language Models are Few-Shot Learners", "Diffusion Models Beat GANs on Image Synthesis"] )

style_input = st.selectbox( "Select Explanation Style", ["Beginner-Friendly", "Technical", "Code-Oriented", "Mathematical"] )

length_input = st.selectbox( "Select Explanation Length", ["Short (1-2 paragraphs)", "Medium (3-5 paragraphs)", "Long (detailed explanation)"] )

template = load_prompt('content/template.json')
if st.button('Summarize'):
  chain = template | model
  result = chain.invoke({
      "paper_input": paper_input,
      "style_input": style_input,
      "length_input": length_input
  })
  st.write(result.content)

In [ ]:
model = ChatOpenAI()
template2 = PromptTemplate(
    template='Greet this person in 5 languages. The name of the person is {name}',
    input_variables=['name']
)
prompt = template2.format({'name': 'Shakib'})
result = model.invoke(prompt)
print(result.content)

In [ ]:
messages = [
    SystemMessage(content='You are a helpful assistant'),
    HumanMessage(content='Tell me about LangChain')
]

result = model.invoke(message)
messages.append(AIMessage(content=result.content))
print(messages)

In [ ]:
chat_template = ChatPromptTemplate.from_messages([
    ('system','You are a helpful customer support agent'),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human','{query}')
])

chat_history = []
with open('chat_history.txt', 'r') as f:
    chat_history.extend(f.readlines())
print(chat_history)

prompt = chat_template.invoke({'chat_history': chat_history, 'query': 'Where is my refund'})
print(prompt)

In [ ]:
chat_history = [
    SystemMessage(content='You are a helpful AI assistant')
]

while True:
  user_input = input('You: ')
  if user_input.lower() == 'exit':
    break
  result = model.invoke(HumanMessage(content=user_input))
  chat_history.append(AIMessage(content=result.content))
  print('AI: ', result.content)

print(chat_history)

**Output and Output-paser**

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser, JsonOutputParser, StructuredOutputParser, ResponseSchema
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

model = ChatOpenAI()
class Review(TypedDict):
  key_themes: Annotated[
      list[str],
      "Write down all the key themes discussed in the review in a list"
  ]
  summary: Annotated[
      str,
      "A brief summary of the review"
  ]
  sentiment: Annotated[
      Literal["positive", "negative", "neutral"],
      "Return sentiment of the review either negative, positive or neutral"
  ]
  pros: Annotated[
      Optional[list[str]],
      "Write down all the pros of the review in a list"
  ]
  cons: Annotated[
      Optional[list[str]],
      "Write down all the cons of the review in a list"
  ]
  name: Annotated[
      Optional[str],
      "Write the name of the reviewer"
  ]

structured_model = model.with_structured_output(Review)
result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful

Review by Nitish Singh
""")

print(result['name'])

In [ ]:
class Review(BaseModel):
  key_themes: list[str] = Field(
      description="Write down all the key themes discussed in the review in a list"
  )
  summary: str = Field(
      description="A brief summary of the review"
  )
  sentiment: Literal["positive", "negative", "neutral"] = Field(
      description="Return sentiment of the review either negative, positive or neutral"
  )
  pros: Optional[list[str]] = Field(
      default=None,
      description="Write down all the pros inside a list",
  )
  cons: Optional[list[str]] = Field(
      default=None,
      description="Write down all the cons inside a list",
  )
  name: Optional[str] = Field(
      default=None,
      description="Write the name of the reviewer"
  )

model = ChatOpenAI()
structured_model = model.with_structured_output(Review)
result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful

Review by Nitish Singh
""")

print(result)

In [ ]:
json_schema = {
  "title": "Review",
  "type": "object",
  "properties": {
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write down all the key themes discussed in the review in a list"
    },
    "summary": {
      "type": "string",
      "description": "A brief summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["pos", "neg"],
      "description": "Return sentiment of the review either negative, positive or neutral"
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the pros inside a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the cons inside a list"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer"
    }
  },
  "required": ["key_themes", "summary", "sentiment"]
}


structured_model = model.with_structured_output(json_schema)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful

Review by Nitish Singh
""")

print(result)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)
class Review(BaseModel):
  key_themes: list[str] = Field(
      description="Write down all the key themes discussed in the review in a list"
  )
  summary: str = Field(
      description="A brief summary of the review"
  )
  sentiment: Literal["positive", "negative", "neutral"] = Field(
      description="Return sentiment of the review either negative, positive or neutral"
  )
  pros: Optional[list[str]] = Field(
      default=None,
      description="Write down all the pros inside a list",
  )
  cons: Optional[list[str]] = Field(
      default=None,
      description="Write down all the cons inside a list",
  )
  name: Optional[str] = Field(
      default=None,
      description="Write the name of the reviewer"
  )

model = ChatOpenAI()
structured_model = model.with_structured_output(Review)
result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful

Review by Nitish Singh
""")

print(result)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)
template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)
template2 = PromptTemplate(
    template='Write a 5 line summary on the following text. /n {text}',
    input_variables=['text']
)

prompt1 = template1.invoke({'topic': 'black hole'})
result = model.invoke(prompt1)
prompt2 = template2.invoke({'text': result.content})
result1 = model.invoke(prompt2)
print(result1.content)

In [ ]:
model = ChatOpenAI()
template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)
template2 = PromptTemplate(
    template='Write a 5 line summary on the following text. /n {text}',
    input_variables=['text']
)

parser = StrOutputParser()
chain = template1 | model | parser | template2 | model
result = chain.invoke({'topic': 'black hole'})
print(result)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)

class Person(BaseModel):
  name: str = Field(description="Name of the person")
  age: int = Field(gt=18, description="Age of the person")
  city: str = Field(description="Name of the city the person belongs to")

parser = PydanticOutputParser(pydantic_object=Person)
template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)
chain = template | model | parser
final_result = chain.invoke({'place': 'Bangladesh'})
print(final_result)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)
parser = JsonOutputParser()
template = PromptTemplate(
    template='Give me 5 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = template | model | parser
final_result = chain.invoke({'topic': 'black hole'})
print(final_result)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)
schema = [
    ResponseSchema(name='fact_1', description='Fact 1 about the topic'),
    ResponseSchema(name='fact_2', description='Fact 2 about the topic'),
    ResponseSchema(name='fact_3', description='Fact 3 about the topic'),
]

parser = StructuredOutputParser.from_response_schemas(schema)
template = PromptTemplate(
    template='Give me 5 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()},
)

chain = template | model | parser
final_result = chain.invoke({'topic': 'black hole'})
print(final_result)

**Chain: Simple, Sequential, Parallel and conditional**

In [ ]:
from langchain.schema.runnable import RunnableParallel, RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser

prompt = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

model = ChatOpenAI()
parser = StrOutputParser()
chain = prompt | model | parser
result = chain.invoke({'topic': 'black hole'})
print(result)
chain.get_graph().print_ascii()

In [ ]:
prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

model = ChatOpenAI()
parser = StrOutputParser()
chain = prompt1 | model | parser | prompt2 | model | parser
result = chain.invoke({'topic': 'black hole'})
print(result)
chain.get_graph().print_ascii

In [ ]:
model1 = ChatOpenAI()
model2 = ChatAnthropic(model_name='claude-3-7-sonnet-20250219')

prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)
prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)
prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parser = StrOutputParser()
parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model1 | parser
chain = merge_chain | merge_chain
text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text':text})
print(result)
chain.get_graph().print_ascii()

In [ ]:
from langchain_core.runnables import RunnableParallel

model = ChatOpenAI()
parser = StrOutputParser()

class Feedback(BaseModel):
  sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)
prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into positive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction': parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2
prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)
prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

input_chain = RunnableParallel({
    "classification": classifier_chain,
    "feedback": lambda x: x["feedback"]
})

branch_chain = RunnableBranch(
    (lambda x: x["classification"].sentiment == 'positive', prompt2 | model | parser),
    (lambda x: x["classification"].sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = input_chain | branch_chain
result = chain.invoke({'feedback': 'I love this product'})
print(result)